# 01 — Loan Default Data Cleaning



Main pipeline policy: include all age and gender groups. Age uses the ordered bands 0–6 with training-fold most-frequent filling; gender uses nominal one-hot encoding. All 148,670 rows remain. The final ensemble uses a development-selected maximum-F1 cutoff. Notebook 09 remains a historical controlled experiment; this notebook is part of the integrated 01–08 run.

## 1. Reproducible setup and data lineage


In [1]:
from pathlib import Path
import hashlib
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'data').exists():
    raise FileNotFoundError('Open this notebook from the Machine_Learning project folder or notebooks folder.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.eda_preprocessing import (
    cleaning_change_log, cleaning_stages, normalise_raw_data, missingness_target_audit,
    data_validity_audit,
    deterministic_clean,
    load_raw_data,
    post_clean_validation,
    save_table,
)

RAW_DATA = PROJECT_ROOT / 'data' / 'raw' / 'Loan_Default.csv'
CLEANED_DATA = PROJECT_ROOT / 'data' / 'processed' / 'loan_default_cleaned.csv'
CLEANING_TABLES = PROJECT_ROOT / 'outputs' / 'tables' / 'cleaning'
CLEANING_TABLES.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 100)
print(f'Project root: {PROJECT_ROOT}')
print(f'Raw input:    {RAW_DATA}')
print(f'Clean output: {CLEANED_DATA}')

Project root: /Users/zichen/Machine_Learning
Raw input:    /Users/zichen/Machine_Learning/data/raw/Loan_Default.csv
Clean output: /Users/zichen/Machine_Learning/data/processed/loan_default_cleaned.csv


## 2. Load and validate the raw dataset


In [2]:
raw = load_raw_data(RAW_DATA)

raw_overview = pd.DataFrame({
    'metric': [
        'rows', 'columns', 'numeric columns', 'categorical columns',
        'missing target values', 'unique target values'
    ],
    'value': [
        len(raw), raw.shape[1], raw.select_dtypes(include=np.number).shape[1],
        raw.select_dtypes(exclude=np.number).shape[1], int(raw['Status'].isna().sum()),
        sorted(raw['Status'].dropna().unique().tolist())
    ]
})
save_table(raw_overview, CLEANING_TABLES / 'raw_overview.csv')
display(raw_overview)



,metric,value
0,rows,148670
1,columns,34
2,numeric columns,13
3,categorical columns,21
4,missing target values,0
5,unique target values,"[0, 1]"


In [3]:
raw_profile = pd.DataFrame({
    'feature': raw.columns,
    'dtype': [str(raw[c].dtype) for c in raw.columns],
    'missing_count': [int(raw[c].isna().sum()) for c in raw.columns],
    'missing_pct': [round(float(raw[c].isna().mean() * 100), 3) for c in raw.columns],
    'unique_including_missing': [int(raw[c].nunique(dropna=False)) for c in raw.columns],
}).sort_values(['missing_pct', 'feature'], ascending=[False, True])

raw_categories = []
for column in raw.select_dtypes(exclude=np.number).columns:
    counts = raw[column].value_counts(dropna=False)
    for category, count in counts.items():
        raw_categories.append({
            'feature': column,
            'category': '<MISSING>' if pd.isna(category) else str(category),
            'count': int(count),
            'percentage': round(float(count / len(raw) * 100), 3),
        })
raw_categories = pd.DataFrame(raw_categories)

duplicate_audit = pd.DataFrame({
    'check': ['exact duplicate rows', 'duplicate IDs', 'duplicates excluding ID'],
    'count': [
        int(raw.duplicated().sum()),
        int(raw['ID'].duplicated().sum()),
        int(raw.drop(columns='ID').duplicated().sum()),
    ],
    'decision': [
        'Remove exact duplicates only if present',
        'Investigate; never drop automatically by ID',
        'Review only because different IDs may represent separate applications',
    ],
})

save_table(raw_profile, CLEANING_TABLES / 'raw_column_profile.csv')
save_table(raw_categories, CLEANING_TABLES / 'raw_category_counts.csv')
save_table(duplicate_audit, CLEANING_TABLES / 'duplicate_audit.csv')

display(raw_profile)
display(duplicate_audit)
display(raw_categories.query("feature in ['Region', 'Security_Type']"))


,feature,dtype,missing_count,missing_pct,unique_including_missing
13,Upfront_charges,float64,39642,26.664,58272
12,Interest_rate_spread,float64,36639,24.645,22517
11,rate_of_interest,float64,36439,24.510,132
33,dtir1,float64,24121,16.225,58
29,LTV,float64,15098,10.155,8485
18,property_value,float64,15098,10.155,386
23,income,float64,9150,6.155,1002
2,loan_limit,object,3344,2.249,3
4,approv_in_adv,object,908,0.611,3
27,age,object,200,0.135,8


,check,count,decision
0,exact duplicate rows,0,Remove exact duplicates only if present
1,duplicate IDs,0,Investigate; never drop automatically by ID
2,duplicates excluding ID,0,Review only because different IDs may represent separate applications


,feature,category,count,percentage
59,Region,North,74722,50.260
60,Region,south,64016,43.059
61,Region,central,8697,5.850
62,Region,North-East,1235,0.831
63,Security_Type,direct,148637,99.978
64,Security_Type,Indriect,33,0.022


## 3. Deterministic cleaning and class balance

Standardise names, whitespace and known category spelling. Check exact duplicates and preserve all 148,670 applications in this dataset, including every recorded gender category. Missing fields and unusual values are preserved for auditing and later preprocessing.

In [4]:
cleaned, cohort_audit = cleaning_stages(raw)
cohort_audit.to_csv(CLEANING_TABLES / 'class_distribution_before_after.csv', index=False)
display(cohort_audit)
assert len(cleaned) == len(raw) == 148_670
assert int(cleaned.status.sum()) == 36_639
assert not raw.duplicated().any() and raw['ID'].is_unique
raw_missingness = missingness_target_audit(normalise_raw_data(raw))
raw_missingness.to_csv(CLEANING_TABLES / 'raw_missingness_target_audit.csv', index=False)
display(raw_missingness)
changes = cleaning_change_log(raw, cleaned)
save_table(changes, CLEANING_TABLES / 'cleaning_change_log.csv')
display(changes)

,stage,rows,columns,non_default,default,default_rate_pct
0,Raw,148670,34,112031,36639,24.644515
1,Cleaned,148670,34,112031,36639,24.644515


,feature,missing_count,missing_pct,default_rate_when_present_pct,default_rate_when_missing_pct,absolute_rate_gap_pp,perfect_separation_warning
4,interest_rate_spread,36639,24.645,0.000,100.000,100.000,True
3,rate_of_interest,36439,24.510,0.178,100.000,99.822,True
5,upfront_charges,39642,26.664,0.140,92.039,91.898,False
8,property_value,15098,10.155,16.128,99.987,83.858,False
12,ltv,15098,10.155,16.128,99.987,83.858,False
10,age,200,0.135,24.543,100.000,75.457,True
11,submission_of_application,200,0.135,24.543,100.000,75.457,True
13,dtir1,24121,16.225,16.322,67.617,51.295,False
6,term,41,0.028,24.641,36.585,11.944,False
9,income,9150,6.155,25.373,13.541,11.832,False


,cleaning_action,items_changed,reason,risk_or_limitation
0,Standardise column names to snake_case and correct neg_amortization spelling,13,"Consistent, reproducible feature naming","Names change, but values do not"
1,Strip leading/trailing whitespace from categorical values,0,Prevent accidental duplicate categories,Internal whitespace is intentionally preserved
2,Standardise south/central region capitalisation,72713,Merge labels that differ only by case,Assumes case has no semantic meaning
3,Correct Indriect security label to Indirect,33,Treat apparent spelling error as one category,Assumption should be confirmed against source documentation
4,Remove exact duplicate rows,0,Prevent repeated identical observations,No deduplication is performed on ID alone


## 4. Validity and suspicious-value audit


In [5]:
validity = data_validity_audit(cleaned)
save_table(validity, CLEANING_TABLES / 'validity_audit.csv')
display(validity)


,rule,severity,flagged_count,flagged_pct,recommended_action
0,numeric_infinite,Impossible,0,0.000,Investigate source or set to missing only with evidence
1,loan_amount_non_positive,Impossible,0,0.000,Investigate source; loan amount should be positive
2,term_non_positive,Impossible,0,0.000,Investigate source; term should be positive
3,credit_score_outside_300_900,Potentially impossible,0,0.000,Confirm scoring scale with source
4,negative_income,Potentially impossible,0,0.000,Confirm units and definition
5,zero_income,Suspicious but plausible,1260,0.848,Retain; examine missingness/zero semantics
6,non_positive_interest_rate,Suspicious,1,0.001,Pricing field excluded from cleaned cohort
7,non_positive_property_value,Potentially impossible,0,0.000,Confirm source; do not repair automatically
8,ltv_above_200_pct,Extreme / possible data issue,17,0.011,Investigate individually; do not remove by IQR alone
9,dti_outside_0_100,Potentially impossible,0,0.000,Confirm ratio definition and range


## 5. Post-cleaning validation gate


In [6]:
validation = post_clean_validation(raw, cleaned)
save_table(validation, CLEANING_TABLES / 'post_clean_validation.csv')
display(validation)

failed = validation.loc[~validation['passed']]
assert failed.empty, f'Cleaning validation failed:\n{failed.to_string(index=False)}'
print('All post-cleaning validation checks passed.')


,validation_check,passed,expected,observed
0,Expected row count after deterministic cleaning,True,148670,148670
1,Column names are unique,True,True,True
2,Target has no missing values,True,0,0
3,Target remains binary 0/1,True,"{0, 1}","[0, 1]"
4,Identifier has no missing values,True,0,0
5,Identifier remains unique,True,True,True
6,No exact duplicates remain,True,0,0
7,No leading/trailing categorical whitespace remains,True,0,0
8,Inconsistent region labels removed,True,0,0
9,Misspelled security label removed,True,0,0


All post-cleaning validation checks passed.


In [7]:
before_after = pd.DataFrame({
    'metric': ['rows', 'columns', 'exact duplicates', 'missing predictor cells', 'missing target', 'default cases', 'non-default cases'],
    'raw': [
        len(raw), raw.shape[1], int(raw.duplicated().sum()), int(raw.drop(columns='Status').isna().sum().sum()), int(raw['Status'].isna().sum()),
        int(raw['Status'].eq(1).sum()), int(raw['Status'].eq(0).sum())
    ],
    'cleaned': [
        len(cleaned), cleaned.shape[1], int(cleaned.duplicated().sum()), int(cleaned.drop(columns='status').isna().sum().sum()), int(cleaned['status'].isna().sum()),
        int(cleaned['status'].eq(1).sum()), int(cleaned['status'].eq(0).sum())
    ]
})
before_after['difference'] = before_after['cleaned'] - before_after['raw']
save_table(before_after, CLEANING_TABLES / 'before_after_summary.csv')
display(before_after)


,metric,raw,cleaned,difference
0,rows,148670,148670,0
1,columns,34,34,0
2,exact duplicates,0,0,0
3,missing predictor cells,181135,181135,0
4,missing target,0,0,0
5,default cases,36639,36639,0
6,non-default cases,112031,112031,0


## 6. Save the canonical cleaned dataset


In [8]:
CLEANED_DATA.parent.mkdir(parents=True, exist_ok=True)
cleaned.to_csv(CLEANED_DATA, index=False, encoding='utf-8')

digest = hashlib.sha256(CLEANED_DATA.read_bytes()).hexdigest()
export_manifest = pd.DataFrame({
    'field': ['output_path', 'rows', 'columns', 'sha256', 'source_file', 'raw_sha256', 'model_training_performed'],
    'value': [str(CLEANED_DATA.relative_to(PROJECT_ROOT)), len(cleaned), cleaned.shape[1], digest, str(RAW_DATA.relative_to(PROJECT_ROOT)), hashlib.sha256(RAW_DATA.read_bytes()).hexdigest(), False]
})
save_table(export_manifest, CLEANING_TABLES / 'cleaned_export_manifest.csv')
display(export_manifest)
print('Saved deterministic cleaning output from all applications.')

,field,value
0,output_path,data/processed/loan_default_cleaned.csv
1,rows,148670
2,columns,34
3,sha256,c1975912253142ef42ed15301e2ace013010a1d2ee1e06dda94fbd7f525b01ff
4,source_file,data/raw/Loan_Default.csv
5,raw_sha256,4234b122f463ff4d563de600ade5ec347a9ab8f02cfc204535f7c0d4929bfe70
6,model_training_performed,False


Saved deterministic cleaning output from all applications.
